# Coding Deep Agent
Dieser "Agent" ist ein Langchain Subgraph, der Teil der **implement_transformation** Node ist. Der Coding Deep Agent verfügt dabei über mehrere Komponenten, wie ein isolierter Arbeitsbereich, optional die Erweiterung durch Skills und Memory Dateien.

## Input
Als Eingabe erhält dieser eine spezifische Aufgabe und den ausformulierten Plan zur Transformation.

## Aufgaben
Der Agent soll dann die Implementierungsschritte im Transformationsplan ausführen und zusätzlich die spezielle Aufgabe des Users beachten.

Die Middleware des Agenten trackt dann alle geschriebenen Dateien, damit diese nachher validiert werden können.

In [1]:
from pathlib import Path
import sys
import logging

logging.basicConfig(level=logging.INFO)

ROOT = Path.cwd().parent.parent
logging.info(f"Adding {ROOT} to Python path for imports in kernel.")
sys.path.insert(0, str(ROOT))  # ROOT, nicht SRC!

# Load config with changed env path
from bxagent.config import Config
config = Config.get_instance(env_path=ROOT / ".env")

INFO:root:Adding /Users/lukas/Masterarbeit/agents/bxAgent to Python path for imports in kernel.


In [2]:
from bxagent.agents.coding.agent import build_coding_deep_agent

agent = build_coding_deep_agent()

Anschließend muss der Prompt zusammengecraftet werden. Hierfür gibt es im `tools/coding` schon den passenden Input Prompt

In [4]:
from bxagent.tools.transformation.plan import (
    TransformationPlan,
    FileTransformationPlanParser,
)

file_plan_parser = FileTransformationPlanParser(
    config.WORKSPACE.PATH /  "TRANSFORMATION.md"
)

path = ROOT / "templates" / "transformation_plan.jinja"
logging.info(f"Checking if template exists at {path}: {path.exists()}")

t = TransformationPlan(parser=file_plan_parser, template_path=ROOT / "templates")

t.update_package_information(
    "de.hof-university.models.Family", "de.hof-university.models.Person"
)
t.update_transformation_direction("Bidirectional transformation")
t.update_model_implementation(
    source_model_implementation="Assume that the Family class is implemented with multiple Members, each have a surname and name",
    target_model_implementation="Assume that the Person class is implemented with just a name field",
)
t.update_transformation_difficulties("1. *Information Loss*: If the family member is mapped to a person, we lose information about the family structure and the correct name split.")
t.update_implementation_steps("""
                              1. Implement a method to transform a Family instance to a Person instance using the Eclipse Modeling Framework
                              2. Implement a method to transform a Person instance back to a Family instance, ensuring that the family structure is preserved as much as possible.
                              """)

INFO:root:Checking if template exists at /Users/lukas/Masterarbeit/agents/bxAgent/templates/transformation_plan.jinja: True


Error occurred while parsing transformation plan: Transformation plan file not found at /Users/lukas/Masterarbeit/agents/bxAgent/.bx-agent-workspace/TRANSFORMATION.md


In [5]:
from bxagent.tools.coding.implement_transformation import create_input_prompt

input_prompt = create_input_prompt(
    task_specification="Implement the transformation between the Family and Person class using the Eclipse Modeling Framework. Do not use the Notification Mechanism!",
    transformation_plan=t,
)

Danach kann der Agent auch schon invoked werden.

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(input={"messages": [HumanMessage(content=input_prompt)]}, version="v2")

INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
